In [2]:
import pandas as pd
import numpy as np

In [3]:
data=pd.read_csv(r"../../data/processed/data_final_shift.csv")
data.head()

,Date,Shanghai Comp,KODEX 200,TOPIX,NASDAQ,KOSDAQ,Brent Crude Oil,Gold Spot,JPY/KRW,USD/KRW,...,GJR_VaR_5_t1,Signal1_Buy,Signal1_Sell,Signal2_Buy,Signal2_Sell,Signal3_Buy,Signal3_Sell,Signal4_Buy,Signal4_Sell,Risk_Label
0,2009-04-17,2503.935059,17370.0,875.0,1673.069946,483.799988,51.959999,867.400024,13.371,1325.800049,...,-2.736404,0,0,0,0,0,0,0,0,Low Risk
1,2009-04-20,2557.456055,17480.0,876.0,1608.209961,491.940002,51.959999,887.000000,13.536,1327.500000,...,-2.641002,0,0,0,0,0,0,0,0,Low Risk
2,2009-04-21,2535.827881,17480.0,855.0,1643.849976,497.190002,51.959999,882.099976,13.727,1354.300049,...,-2.553085,0,0,0,0,0,0,0,0,Low Risk
3,2009-04-22,2461.345947,17715.0,856.0,1646.119995,509.899994,51.959999,891.799988,13.726,1346.599976,...,-2.469281,0,0,0,0,0,0,0,0,Low Risk
4,2009-04-23,2463.954102,17895.0,862.0,1652.209961,514.090027,51.959999,905.900024,13.618,1333.599976,...,-2.386329,0,0,0,0,0,1,0,0,Low Risk


In [4]:
# Label을 0/1로 변환 (Low risk=0, High risk=1)
label_norm = data['Risk_Label'].astype(str).str.strip().str.lower()
label_map = {'low risk': 0, 'high risk': 1}
data['Risk_Label'] = label_norm.map(label_map)

data['Risk_Label'] = data['Risk_Label'].astype(int)

# train:valid:test 5:3:2
data_train = data[:int(len(data) * 0.5)]

# X, y 분리
X_train = data_train.drop(['Risk_Label', 'Date'], axis=1)  # 설명변수, Date는 ML에서 불필요
y_train = data_train['Risk_Label']  # 반응변수

In [5]:
def get_rolling_windows(df, window_size, step):
    windows = []
    for start in range(0, len(df) - window_size + 1, step):
        end = start + window_size
        windows.append((start, end))

        last_start = len(df) - window_size
    last_end = len(df)

    if windows[-1] != (last_start, last_end):
        windows.append((last_start, last_end))

    return windows

In [ ]:
from boruta import BorutaPy
from sklearn.ensemble import RandomForestClassifier

def run_boruta(X, y):
    rf = RandomForestClassifier(
       n_jobs=-1,                  #병렬 처리
        class_weight='balanced',            #클래스 불균형 처리(vs 'balanced')
        max_depth=10,             #트리 깊이 제한
        random_state=1              #seed 고정
    )

    boruta = BorutaPy(
        estimator=rf,
        n_estimators='auto',
        perc=80,
        alpha=0.05,
        max_iter=50,
        random_state=1,
        verbose=2
    )

    boruta.fit(X.values, y.values)

    return boruta.support_, boruta.support_weak_
#=========================================
#보루타 적용

window_size = 750   # 적절히 조정
step = 125

windows = get_rolling_windows(data_train, window_size, step)

feature_names = X_train.columns
selection_matrix = []
tentative_matrix = []

for start, end in windows:
    X_w = X_train.iloc[start:end]
    y_w = y_train.iloc[start:end]

    selected, tentative = run_boruta(X_w, y_w)

    selection_matrix.append(selected)
    tentative_matrix.append(tentative)

selection_matrix = np.array(selection_matrix)
tentative_matrix = np.array(tentative_matrix)

Iteration: 	1 / 50
Confirmed: 	0
Tentative: 	46
Rejected: 	0
Iteration: 	2 / 50
Confirmed: 	0
Tentative: 	46
Rejected: 	0
Iteration: 	3 / 50
Confirmed: 	0
Tentative: 	46
Rejected: 	0
Iteration: 	4 / 50
Confirmed: 	0
Tentative: 	46
Rejected: 	0
Iteration: 	5 / 50
Confirmed: 	0
Tentative: 	46
Rejected: 	0
Iteration: 	6 / 50
Confirmed: 	0
Tentative: 	46
Rejected: 	0
Iteration: 	7 / 50
Confirmed: 	0
Tentative: 	46
Rejected: 	0
Iteration: 	8 / 50
Confirmed: 	0
Tentative: 	13
Rejected: 	33
Iteration: 	9 / 50
Confirmed: 	3
Tentative: 	10
Rejected: 	33
Iteration: 	10 / 50
Confirmed: 	3
Tentative: 	10
Rejected: 	33
Iteration: 	11 / 50
Confirmed: 	3
Tentative: 	10
Rejected: 	33
Iteration: 	12 / 50
Confirmed: 	3
Tentative: 	8
Rejected: 	35
Iteration: 	13 / 50
Confirmed: 	3
Tentative: 	8
Rejected: 	35
Iteration: 	14 / 50
Confirmed: 	3
Tentative: 	8
Rejected: 	35
Iteration: 	15 / 50
Confirmed: 	3
Tentative: 	8
Rejected: 	35
Iteration: 	16 / 50
Confirmed: 	3
Tentative: 	8
Rejected: 	35
Iteration: 	1

In [7]:
num_windows = selection_matrix.shape[0]

# 최근일수록 weight 크게
weights = np.exp(np.linspace(-1, 0, num_windows))

# 정규화
weights = weights / weights.sum()

# weighted score 계산
confirmed_scores = selection_matrix.T @ weights
tentative_scores = tentative_matrix.T @ weights
total_scores = confirmed_scores + 0.5 * tentative_scores

In [8]:
score_table = pd.DataFrame({
    "feature": feature_names,
    "score": total_scores
}).sort_values("score", ascending=False)

print(score_table.head(30))

                         feature     score
28               KOSPI 200_PDI14  1.000000
15                     return(%)  0.788017
18                  VKOSPI_Close  0.432796
3                         NASDAQ  0.388616
30               KOSPI 200_DMI14  0.211983
33                   GARCH_mu_t1  0.142361
14              KOSPI 200 Volume  0.116555
29               KOSPI 200_MDI14  0.095427
9                        CNY/KRW  0.095427
7                        JPY/KRW  0.071180
16  KOSPI 200 lagged_1_return(%)  0.071180
31               KOSPI 200_ADX14  0.000000
32               KOSPI 200_ROC12  0.000000
34                GARCH_sigma_t1  0.000000
35                  GJR_sigma_t1  0.000000
0                  Shanghai Comp  0.000000
27          KOSPI 200_BB_LOWER15  0.000000
37                  GJR_VaR_5_t1  0.000000
38                   Signal1_Buy  0.000000
39                  Signal1_Sell  0.000000
40                   Signal2_Buy  0.000000
41                  Signal2_Sell  0.000000
42         

In [ ]:
selected_features = score_table.head(7)["feature"].tolist()

selected_data = data.loc[:, ['Date']+selected_features + ["Risk_Label"]].copy()
selected_data 

,Date,KOSPI 200_PDI14,return(%),VKOSPI_Close,NASDAQ,KOSPI 200_DMI14,GARCH_mu_t1,KOSPI 200 Volume,Risk_Label
0,2009-04-17,34.513789,-0.233192,35.49,1673.069946,14.647201,-0.077605,251600000.0,0
1,2009-04-20,32.336590,0.564563,36.15,1608.209961,12.666581,-0.074431,183300000.0,0
2,2009-04-21,29.950904,-0.197523,36.40,1643.849976,9.561273,-0.073820,204200000.0,0
3,2009-04-22,32.819567,1.408955,35.01,1646.119995,13.493950,-0.067794,291400000.0,0
4,2009-04-23,33.860892,0.992765,33.39,1652.209961,15.409494,-0.062765,277400000.0,0
...,...,...,...,...,...,...,...,...,...
4103,2026-02-20,43.843336,2.276801,43.87,22886.070312,26.045709,0.048244,352700000.0,0
4104,2026-02-23,46.773163,0.684025,46.34,22627.269531,30.226272,0.048309,356700000.0,0
4105,2026-02-24,44.450274,2.441388,48.12,22863.679688,29.474786,0.048395,253500000.0,0
4106,2026-02-25,48.833464,1.893162,49.57,23152.080078,35.045213,0.048487,348800000.0,0


In [18]:
#selected_data의 분산팽창계수 데이터 프레임으로 확인 코드
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif_data = pd.DataFrame()
vif_data["feature"] = selected_data.columns
vif_data["VIF"] = [variance_inflation_factor(selected_data.values, i) for i in range(selected_data.shape[1])]
print(vif_data)


            feature        VIF
0   KOSPI 200_PDI14  17.012246
1         return(%)   1.077725
2      VKOSPI_Close  10.480755
3            NASDAQ   3.714581
4   KOSPI 200_DMI14   2.509601
5       GARCH_mu_t1   5.406964
6  KOSPI 200 Volume   1.012648
7        Risk_Label   1.152861


In [19]:
selected_data

,KOSPI 200_PDI14,return(%),VKOSPI_Close,NASDAQ,KOSPI 200_DMI14,GARCH_mu_t1,KOSPI 200 Volume,Risk_Label
0,34.513789,-0.233192,35.49,1673.069946,14.647201,-0.077605,251600000.0,0
1,32.336590,0.564563,36.15,1608.209961,12.666581,-0.074431,183300000.0,0
2,29.950904,-0.197523,36.40,1643.849976,9.561273,-0.073820,204200000.0,0
3,32.819567,1.408955,35.01,1646.119995,13.493950,-0.067794,291400000.0,0
4,33.860892,0.992765,33.39,1652.209961,15.409494,-0.062765,277400000.0,0
...,...,...,...,...,...,...,...,...
4103,43.843336,2.276801,43.87,22886.070312,26.045709,0.048244,352700000.0,0
4104,46.773163,0.684025,46.34,22627.269531,30.226272,0.048309,356700000.0,0
4105,44.450274,2.441388,48.12,22863.679688,29.474786,0.048395,253500000.0,0
4106,48.833464,1.893162,49.57,23152.080078,35.045213,0.048487,348800000.0,0


In [24]:
selected_data.to_csv(r'..\..\data\processed\data_selected_winBoruta.csv', index=False,encoding="utf-8-sig")